# Method 5 v2: EfficientNet-B0 + CBAM Attention + RAF-DB (FIXED)
---
## Cac bug da fix so voi v1:
| Bug | Mo ta | Fix |
|-----|-------|-----|
| **[CRITICAL] Wrong preprocessing** | Dung `rescale=1/255` thay vi `preprocess_input` | Dung `efficientnet.preprocess_input` |
| **Input size 100x100** | Feature map chi con 4x4, CBAM kem hieu qua | Tang len **224x224** (chuan EfficientNetB0) |
| **Training instability** | Val acc nhan qua: 2%->20%->5%->39% | Giam label_smoothing, can chinh optimizer |
| **Phase 2 bat dau sai** | Load lai best model truoc khi fine-tune | Dam bao weights tu Phase 1 duoc giu |

**Muc tieu:** Accuracy **>= 85%** tren RAF-DB test set

In [ ]:
# ================================================
# CELL 1: SETUP & GPU CHECK
# ================================================
!nvidia-smi

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {gpus}')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU ENABLED!')
else:
    print('NO GPU! Go to Runtime > Change runtime type > GPU')

In [ ]:
# ================================================
# CELL 2: IMPORT LIBRARIES
# ================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import json
import shutil
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
)
from tensorflow.keras.applications import EfficientNetB0
# FIX 1: Import dung preprocessing function cho EfficientNet
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('Libraries imported!')
print(f'[FIX] Using efficientnet.preprocess_input (NOT rescale=1/255)')

In [ ]:
# ================================================
# CELL 3: MOUNT DRIVE & COPY DATASET TO LOCAL
# ================================================
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

DRIVE_DATASET = '/content/drive/MyDrive/CaptoneProject/dataset_raf_db'
LOCAL_PATH = '/content/dataset_rafdb'

possible = [
    ('DATASET/train', 'DATASET/test'),
    ('train', 'test'),
]

DRIVE_TRAIN = None
DRIVE_TEST = None
for ts, es in possible:
    t = os.path.join(DRIVE_DATASET, ts)
    e = os.path.join(DRIVE_DATASET, es)
    if os.path.exists(t) and os.path.exists(e):
        DRIVE_TRAIN = t
        DRIVE_TEST = e
        print(f'Tim thay dataset tren Drive!')
        print(f'  Train: {DRIVE_TRAIN}')
        print(f'  Test:  {DRIVE_TEST}')
        break

if DRIVE_TRAIN is None:
    for root, dirs, files in os.walk(DRIVE_DATASET):
        if set(['1','2','3','4','5','6','7']).issubset(set(dirs)):
            print(f'Tim thay folder chua 7 class tai: {root}')
            break
    print('Kiem tra lai cau truc folder tren Drive!')
else:
    TRAIN_DIR = os.path.join(LOCAL_PATH, 'train')
    TEST_DIR = os.path.join(LOCAL_PATH, 'test')

    if not os.path.exists(TRAIN_DIR):
        print('\nCopying dataset tu Drive sang local (1-3 phut)...')
        os.makedirs(LOCAL_PATH, exist_ok=True)

        print('  Copying train set...')
        shutil.copytree(DRIVE_TRAIN, TRAIN_DIR)
        tc = sum([len(f) for _, _, f in os.walk(TRAIN_DIR)])
        print(f'    -> {tc} images')

        print('  Copying test set...')
        shutil.copytree(DRIVE_TEST, TEST_DIR)
        ec = sum([len(f) for _, _, f in os.walk(TEST_DIR)])
        print(f'    -> {ec} images')
        print('Dataset copied!')
    else:
        tc = sum([len(f) for _, _, f in os.walk(TRAIN_DIR)])
        ec = sum([len(f) for _, _, f in os.walk(TEST_DIR)])
        print('Dataset da co san local!')

    print(f'\nTRAIN_DIR: {TRAIN_DIR} ({tc} images)')
    print(f'TEST_DIR:  {TEST_DIR} ({ec} images)')
    print(f'Classes: {sorted(os.listdir(TRAIN_DIR))}')

In [ ]:
# ================================================
# CELL 4: CONFIGURATION
# FIX 2: IMG_SIZE = 224 (EfficientNetB0 standard input)
# ================================================

# FIX: Tang size len 224 - EfficientNetB0 duoc train voi 224x224
# 100x100 -> feature map 4x4 (qua nho!)
# 224x224 -> feature map 7x7 (tot hon nhieu)
IMG_SIZE        = 224  # FIXED: was 100
BATCH_SIZE      = 32
EPOCHS_PHASE1   = 20   # Tang them mot chut
EPOCHS_PHASE2   = 60
NUM_CLASSES     = 7
SEED            = 42
LABEL_SMOOTHING = 0.05  # FIXED: Giam tu 0.1 xuong 0.05

# Class mapping: folder 1-7 -> Surprise, Fear, Disgust, Happiness, Sadness, Anger, Neutral
EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

np.random.seed(SEED)
tf.random.set_seed(SEED)

CHECKPOINT_DIR = '/content/drive/MyDrive/CaptoneProject/checkpoints/method5_rafdb_v2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = f'{CHECKPOINT_DIR}/best_model.keras'
HISTORY_PATH = f'{CHECKPOINT_DIR}/history.pkl'

print('Config set!')
print(f'  IMG_SIZE={IMG_SIZE} (FIX: 100->224), BATCH={BATCH_SIZE}')
print(f'  Phase 1: {EPOCHS_PHASE1} epochs | Phase 2: {EPOCHS_PHASE2} epochs')
print(f'  Label Smoothing: {LABEL_SMOOTHING} (FIX: 0.1->0.05)')
print(f'  Emotions: {EMOTIONS}')

In [ ]:
# ================================================
# CELL 5: DATA GENERATORS - FIX PREPROCESSING
# FIX 3: Dung preprocess_input thay vi rescale=1/255
# EfficientNet expect pixel [0,255] -> normalize internally
# ================================================

# FIX: preprocess_input chuyen [0,255] -> [-1,1] hoac [0,1] theo cach EfficientNet can
# KHONG dung rescale=1./255 nua!
train_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess,  # FIX: thay rescale
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.15
)

# FIX: Test generator cung phai dung preprocess_input
test_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess  # FIX: thay rescale
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'\n[FIX] Preprocessing: efficientnet_preprocess (NOT rescale=1/255)')
print(f'[FIX] Input size: {IMG_SIZE}x{IMG_SIZE} (NOT 100x100)')
print(f'\nData Ready!')
print(f'  Train: {train_generator.samples} images')
print(f'  Val:   {validation_generator.samples} images')
print(f'  Test:  {test_generator.samples} images')
print(f'  Classes: {train_generator.class_indices}')

In [ ]:
# ================================================
# CELL 6: CLASS WEIGHTS, FOCAL LOSS & IMBALANCE ANALYSIS
# WARNING 1: class_weight qua cao (>5x) co the gay:
#   - gradient oscillation
#   - overcompensation -> model bo qua majority class
# SOLUTION: Kiem tra imbalance ratio, co option Focal Loss
# ================================================

# ---------- Focal Loss (thay the class_weight neu can) ----------
class FocalLoss(keras.losses.Loss):
    """Focal Loss tu dong xu ly imbalance - khong can class_weight.
    gamma=2.0: tap trung vao hard examples
    alpha=0.25: base weight cho positive class
    """
    def __init__(self, gamma=2.0, alpha=0.25, label_smoothing=0.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        if self.label_smoothing > 0:
            n_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1 - self.label_smoothing) + self.label_smoothing / n_classes
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce     = -y_true * tf.math.log(y_pred)
        weight = self.alpha * y_true * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha,
                    'label_smoothing': self.label_smoothing})
        return cfg

# ---------- Tinh class weights ----------
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    'balanced', classes=np.unique(train_labels), y=train_labels
)
class_weights = dict(enumerate(class_weights_array))

# ---------- Kiem tra imbalance ratio ----------
max_w          = max(class_weights_array)
min_w          = min(class_weights_array)
imbalance_ratio = max_w / min_w

print('=== Imbalance Analysis ===')
print(f'Imbalance ratio (max_weight / min_weight): {imbalance_ratio:.2f}x')

# RAF-DB: Fear=239 anh, Happiness=4057 anh -> ratio ~17x -> weight Fear=6.24
# ⚠ Neu ratio > 5: class_weight co the gay gradient oscillation!
# ⚠ Neu ratio > 10: STRONGLY recommend Focal Loss

# ---------- CONFIG: Thay doi 2 flag nay de thu nghiem ----------
USE_CLASS_WEIGHT = True   # False: bo class_weight hoan toan
USE_FOCAL_LOSS   = False  # True : dung Focal Loss thay CrossEntropy
                          # (Focal Loss tu xu ly imbalance, KHONG can class_weight)

if imbalance_ratio > 10:
    print(f'[WARNING] Ratio {imbalance_ratio:.1f}x > 10x - STRONGLY recommend USE_FOCAL_LOSS=True')
elif imbalance_ratio > 5:
    print(f'[WARNING] Ratio {imbalance_ratio:.1f}x > 5x - class_weight co the gay oscillation')
    print('[TIP]    Neu Phase 1 val_acc < 75%, dat USE_FOCAL_LOSS=True va chay lai')
else:
    print(f'[OK]     Ratio {imbalance_ratio:.1f}x <= 5x - class_weight an toan')

print(f'\nSetting: USE_CLASS_WEIGHT={USE_CLASS_WEIGHT}, USE_FOCAL_LOSS={USE_FOCAL_LOSS}')
print()
print('Class Weights:')
for i, emotion in enumerate(EMOTIONS):
    count = np.sum(train_labels == i)
    flag = ' <- HIGH!' if class_weights[i] > 4 else ''
    print(f'  {emotion:12s}: {count:5d} images -> weight = {class_weights[i]:.4f}{flag}')

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts_train = [np.sum(train_labels == i) for i in range(NUM_CLASSES)]
colors = ['#FF6B6B', '#FFE66D', '#4ECDC4', '#45B7D1', '#96CEB4', '#FF8C94', '#A8E6CF']

axes[0].barh(EMOTIONS, counts_train, color=colors)
axes[0].set_title('Train Set Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(counts_train):
    axes[0].text(v + 20, i, str(v), va='center', fontweight='bold')

test_labels_arr = test_generator.classes
counts_test   = [np.sum(test_labels_arr == i) for i in range(NUM_CLASSES)]
axes[1].barh(EMOTIONS, counts_test, color=colors)
axes[1].set_title('Test Set Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(counts_test):
    axes[1].text(v + 5, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 7: VISUALIZE SAMPLE IMAGES
# ================================================
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle('RAF-DB Sample Images', fontsize=16, fontweight='bold')

for class_idx in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_idx + 1))
    if os.path.exists(class_dir):
        images = sorted(os.listdir(class_dir))
        for row in range(2):
            if row < len(images):
                img = plt.imread(os.path.join(class_dir, images[row]))
                axes[row, class_idx].imshow(img)
            axes[row, class_idx].axis('off')
            if row == 0:
                axes[row, class_idx].set_title(EMOTIONS[class_idx], fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/sample_images.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 8: CBAM ATTENTION BLOCK
# ================================================
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        ch = input_shape[-1]
        self.dense1 = layers.Dense(ch // self.ratio, activation='relu',
                                   kernel_initializer='he_normal')
        self.dense2 = layers.Dense(ch, kernel_initializer='he_normal')
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        super().build(input_shape)

    def call(self, x):
        ch = tf.shape(x)[-1]
        avg = self.dense2(self.dense1(self.gap(x)))
        mx  = self.dense2(self.dense1(self.gmp(x)))
        att = tf.sigmoid(avg + mx)
        return x * tf.reshape(att, (-1, 1, 1, tf.shape(att)[-1]))

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same',
                                  activation='sigmoid',
                                  kernel_initializer='he_normal')
        super().build(input_shape)

    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx  = tf.reduce_max(x, axis=-1, keepdims=True)
        att = self.conv(tf.concat([avg, mx], axis=-1))
        return x * att

    def get_config(self):
        config = super().get_config()
        config.update({'kernel_size': self.kernel_size})
        return config


class CBAMBlock(layers.Layer):
    def __init__(self, ratio=8, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.ca = ChannelAttention(ratio=ratio)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def call(self, x):
        return self.sa(self.ca(x))

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio, 'kernel_size': self.kernel_size})
        return config

print('CBAM Attention Block defined!')

In [ ]:
# ================================================
# CELL 9: BUILD MODEL - EfficientNet-B0 + CBAM
# FIX: Input 224x224 cho feature map 7x7 (thay vi 4x4)
# ================================================
def build_efficientnet_cbam(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=7):
    base_model = EfficientNetB0(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Phase 1: Freeze backbone

    inputs = layers.Input(shape=input_shape)
    # FIX: training=False rat quan trong khi backbone frozen
    x = base_model(inputs, training=False)
    # With 224x224: feature map = 7x7x1280 (tot hon 4x4x1280)

    # CBAM Attention - hieu qua hon voi feature map lon hon
    x = CBAMBlock(ratio=16, kernel_size=7)(x)

    x = layers.GlobalAveragePooling2D()(x)

    # Classifier Head - dung Dropout nhe hon
    x = layers.Dense(512, kernel_regularizer=regularizers.l2(0.0001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)  # FIX: Giam dropout tu 0.5 xuong 0.4

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(0.0001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)  # FIX: Giam dropout tu 0.5 xuong 0.3

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model, base_model

model, base_model = build_efficientnet_cbam()
model.summary()
print(f'\nModel built!')
print(f'  Total params:     {model.count_params():,}')
print(f'  Input shape:      {model.input_shape}')
print(f'  Feature map size: 7x7x1280 (FIX: was 4x4x1280 with 100x100)')

In [ ]:
# ================================================
# CELL 10: PHASE 1 - TRAIN HEAD (Backbone Frozen)
# ⚠ Dung class_weight co dieu kien dua tren USE_CLASS_WEIGHT
# ⚠ Ho tro Focal Loss thay the neu USE_FOCAL_LOSS=True
# ================================================
steps_per_epoch = train_generator.samples // BATCH_SIZE

# Chon loss function
if USE_FOCAL_LOSS:
    loss_fn    = FocalLoss(gamma=2.0, alpha=0.25, label_smoothing=LABEL_SMOOTHING)
    cw_phase1  = None   # Focal Loss tu xu ly imbalance
    print('[INFO] Dung Focal Loss (gamma=2) - class_weight=None')
else:
    loss_fn   = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
    cw_phase1 = class_weights if USE_CLASS_WEIGHT else None
    print(f'[INFO] CrossEntropy(ls={LABEL_SMOOTHING}) + class_weight={USE_CLASS_WEIGHT}')

# Phase 1: Adam + ReduceLROnPlateau (on dinh hon CosineDecay)
optimizer_p1 = keras.optimizers.Adam(learning_rate=1e-3)
model.compile(optimizer=optimizer_p1, loss=loss_fn, metrics=['accuracy'])

callbacks_p1 = [
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3,
                      min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=7,
                  restore_best_weights=True, verbose=1)
]

print('=' * 55)
print('PHASE 1: TRAIN HEAD ONLY (Backbone Frozen)')
print(f'  Loss:         {"FocalLoss(gamma=2)" if USE_FOCAL_LOSS else "CrossEntropy+LS"}')
print(f'  ClassWeight:  {cw_phase1 is not None}')
print(f'  Optimizer:    Adam(lr=1e-3) + ReduceLROnPlateau')
print(f'  Epochs:       {EPOCHS_PHASE1} (EarlyStopping patience=7)')
print(f'  Input:        {IMG_SIZE}x{IMG_SIZE} + efficientnet_preprocess')
print('=' * 55)

history1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE1,
    validation_data=validation_generator,
    callbacks=callbacks_p1,
    class_weight=cw_phase1,
    verbose=1
)

p1_best = max(history1.history['val_accuracy'])
print(f'\nPhase 1 Done! Best Val Acc: {p1_best*100:.2f}%')
if p1_best < 0.75:
    print(f'[WARNING] Val acc = {p1_best*100:.1f}% < 75%!')
    print('[ACTION]  Quay lai Cell 6, dat USE_FOCAL_LOSS=True roi chay lai tu Cell 6')
else:
    print(f'[OK] Phase 1 on track -> chuan bi fine-tune backbone!')

In [ ]:
# ================================================
# CELL 11: PHASE 2 - FINE-TUNE BACKBONE
# ⚠ WARNING 2: BatchNorm PHAI frozen khi unfreeze backbone!
#   Neu BatchNorm trainable=True:
#   - Running mean/variance bi pha huy
#   - Accuracy tut manh va unstable
# -> Code nay da co verify + logging de dam bao dung
# ================================================
print('Loading best Phase 1 model...')
custom_objs = {
    'CBAMBlock': CBAMBlock, 'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention, 'FocalLoss': FocalLoss
}
model = keras.models.load_model(BEST_MODEL_PATH, custom_objects=custom_objs)
print(f'Loaded! Phase 1 best val_acc = {p1_best*100:.2f}%')

# Lay lai base_model
base_model_loaded = model.layers[1]  # EfficientNetB0
base_model_loaded.trainable = True

# ⚠ STEP 1: Freeze TẤT CẢ BatchNorm layers TRUOC TIEN
#   (phai lam truoc khi set bat ky layer nao khac)
bn_frozen = 0
for layer in base_model_loaded.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        bn_frozen += 1

# ⚠ STEP 2: Freeze cac layer o dau (truoc FINE_TUNE_AT)
total_layers  = len(base_model_loaded.layers)
FINE_TUNE_AT  = total_layers - 100  # Unfreeze 100 layer cuoi
for layer in base_model_loaded.layers[:FINE_TUNE_AT]:
    layer.trainable = False  # Ghi de (BatchNorm da False roi, ok)

# ⚠ STEP 3: VERIFY - dam bao khong co BatchNorm nao trainable
bn_still_trainable = sum(
    1 for l in base_model_loaded.layers
    if isinstance(l, tf.keras.layers.BatchNormalization) and l.trainable
)
if bn_still_trainable > 0:
    raise RuntimeError(f'[ERROR] {bn_still_trainable} BatchNorm layers van con trainable! Dung lai!')
else:
    print(f'[OK] BatchNorm verified: {bn_frozen} layers, TAT CA da frozen')

trainable_count = sum([K.count_params(w) for w in model.trainable_weights])
print(f'[OK] Unfreeze: {total_layers - FINE_TUNE_AT} conv layers (was 50 in v1)')
print(f'[OK] Trainable params: {trainable_count:,}')

# Chon class_weight nhat quan voi Phase 1
cw_phase2 = None if USE_FOCAL_LOSS else (class_weights if USE_CLASS_WEIGHT else None)
print(f'[OK] class_weight Phase 2: {cw_phase2 is not None}')

# Fine-tune voi lr rat nho
cosine_p2 = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=EPOCHS_PHASE2 * steps_per_epoch,
    alpha=0.001
)
optimizer_p2 = keras.optimizers.AdamW(learning_rate=cosine_p2, weight_decay=1e-5)
model.compile(optimizer=optimizer_p2, loss=loss_fn, metrics=['accuracy'])

callbacks_p2 = [
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12,
                  restore_best_weights=True, verbose=1),
]

print('=' * 55)
print('PHASE 2: FINE-TUNE BACKBONE')
print(f'  AdamW + CosineDecay (1e-5 -> ~1e-8)')
print(f'  EarlyStopping patience=12')
print(f'  [VERIFIED] BatchNorm frozen: {bn_frozen} layers')
print(f'  [FIX] Unfrozen conv: {total_layers - FINE_TUNE_AT} layers')
print('=' * 55)

history2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE2,
    validation_data=validation_generator,
    callbacks=callbacks_p2,
    class_weight=cw_phase2,
    verbose=1
)

p2_best = max(history2.history['val_accuracy'])
print(f'\nPhase 2 Done! Best Val Acc: {p2_best*100:.2f}%')

In [ ]:
# ================================================
# CELL 12: EVALUATE ON TEST SET
# ================================================
print('Loading best model...')
model = keras.models.load_model(BEST_MODEL_PATH, custom_objects={
    'CBAMBlock': CBAMBlock,
    'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention
})

test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f'\n{"="*55}')
print(f'TEST ACCURACY: {test_acc*100:.2f}%')
print(f'TEST LOSS:     {test_loss:.4f}')
print(f'{"="*55}')

y_pred = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

print('\n' + classification_report(y_true, y_pred_classes, target_names=EMOTIONS))

# Kiem tra xem model co bi collapse khong
unique_preds = np.unique(y_pred_classes)
print(f'\nSo class duoc predict: {len(unique_preds)}/7')
if len(unique_preds) < 7:
    print(f'[WARNING] Model chi predict {len(unique_preds)} classes: {[EMOTIONS[i] for i in unique_preds]}')
    print('[WARNING] Van con ve van de training!')
else:
    print('[OK] Model dang predict du 7 classes!')

In [ ]:
# ================================================
# CELL 13: CONFUSION MATRIX
# ================================================
cm = confusion_matrix(y_true, y_pred_classes)

# Normalize confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absolute numbers
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Count) - Acc: {test_acc*100:.2f}%',
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[1])
axes[1].set_title(f'Confusion Matrix (Normalized)',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 14: TRAINING HISTORY
# ================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs_range = range(1, len(all_acc) + 1)
p1_end = len(history1.history['accuracy'])

axes[0].plot(epochs_range, all_acc, 'b-', label='Train Acc', linewidth=2)
axes[0].plot(epochs_range, all_val_acc, 'r-', label='Val Acc', linewidth=2)
axes[0].axvline(x=p1_end, color='green', linestyle='--', label='Phase 2 Start', linewidth=2)
axes[0].axhline(y=0.85, color='orange', linestyle=':', label='Target 85%', linewidth=2)
axes[0].set_title('Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

axes[1].plot(epochs_range, all_loss, 'b-', label='Train Loss', linewidth=2)
axes[1].plot(epochs_range, all_val_loss, 'r-', label='Val Loss', linewidth=2)
axes[1].axvline(x=p1_end, color='green', linestyle='--', label='Phase 2 Start', linewidth=2)
axes[1].set_title('Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_history.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 15: TEST TIME AUGMENTATION (TTA)
# FIX: TTA cung phai dung efficientnet_preprocess
# ================================================
def predict_with_tta(model, test_dir, img_size, batch_size, n_aug=10):
    # FIX: Dung preprocess_input cho TTA
    base_gen = ImageDataGenerator(
        preprocessing_function=efficientnet_preprocess
    ).flow_from_directory(
        test_dir, target_size=(img_size, img_size),
        color_mode='rgb', batch_size=batch_size,
        class_mode='categorical', shuffle=False
    )
    preds = model.predict(base_gen, verbose=0)

    for i in range(n_aug):
        aug_gen = ImageDataGenerator(
            preprocessing_function=efficientnet_preprocess,  # FIX
            rotation_range=10,
            width_shift_range=0.05,
            height_shift_range=0.05,
            horizontal_flip=True,
            zoom_range=0.1,
            brightness_range=[0.9, 1.1]
        ).flow_from_directory(
            test_dir, target_size=(img_size, img_size),
            color_mode='rgb', batch_size=batch_size,
            class_mode='categorical', shuffle=False
        )
        preds += model.predict(aug_gen, verbose=0)

    preds /= (n_aug + 1)
    return preds, base_gen.classes

print('Running TTA (10 augmentations)...')
tta_preds, y_true_tta = predict_with_tta(model, TEST_DIR, IMG_SIZE, BATCH_SIZE)
tta_classes = np.argmax(tta_preds, axis=1)
tta_acc = np.mean(tta_classes == y_true_tta)

print(f'\n{"="*55}')
print(f'Standard Accuracy: {test_acc*100:.2f}%')
print(f'TTA Accuracy:      {tta_acc*100:.2f}%')
print(f'TTA Improvement:   +{(tta_acc - test_acc)*100:.2f}%')
print(f'{"="*55}')

In [ ]:
# ================================================
# CELL 16: SAVE MODEL FOR WEBAPP
# ================================================
FINAL_PATH = f'{CHECKPOINT_DIR}/emotion_rafdb_v2_final.keras'
model.save(FINAL_PATH)

WEIGHTS_PATH = f'{CHECKPOINT_DIR}/emotion_rafdb_v2_weights.weights.h5'
model.save_weights(WEIGHTS_PATH)

config = {
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'emotions': EMOTIONS,
    'color_mode': 'rgb',
    'preprocessing': 'efficientnet_preprocess_input',
    'test_accuracy': float(test_acc),
    'tta_accuracy': float(tta_acc),
    'model_name': 'EfficientNetB0_CBAM_RAFDB_v2',
    'fixes': [
        'efficientnet_preprocess_input (not rescale)',
        'input_size_224 (not 100)',
        'unfreeze_100_layers (not 50)',
        'reduced_dropout (0.4/0.3 not 0.5)',
        'label_smoothing_0.05 (not 0.1)'
    ]
}
with open(f'{CHECKPOINT_DIR}/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Luu history
with open(HISTORY_PATH, 'wb') as f:
    pickle.dump({'phase1': history1.history, 'phase2': history2.history}, f)

print('Model saved!')
print(f'  Full:    {FINAL_PATH}')
print(f'  Weights: {WEIGHTS_PATH}')
print(f'  Config:  {CHECKPOINT_DIR}/model_config.json')
print(f'\nIMPORTANT: Khi load model trong webapp, phai dung:')
print(f'  from tensorflow.keras.applications.efficientnet import preprocess_input')
print(f'  img = preprocess_input(img)  # KHONG dung img/255!')

In [ ]:
# ================================================
# CELL 17: FINAL SUMMARY & COMPARISON
# ================================================
print('=' * 65)
print(' SUMMARY: Method 5 v2 - EfficientNet-B0 + CBAM + RAF-DB (FIXED)')
print('=' * 65)
print(f' Dataset:       RAF-DB ({train_generator.samples + test_generator.samples} images)')
print(f' Architecture:  EfficientNet-B0 + CBAM Attention')
print(f' Optimizer:     Adam(P1) + AdamW+CosineDecay(P2)')
print(f' Input:         {IMG_SIZE}x{IMG_SIZE} RGB (FIX: was 100x100)')
print(f' Preprocessing: efficientnet_preprocess_input (FIX: was rescale)')
print(f' Params:        {model.count_params():,}')
print(f'')
print(f' === RESULTS ===')
print(f' Phase 1 Best Val Acc: {p1_best*100:.2f}%')
print(f' Phase 2 Best Val Acc: {p2_best*100:.2f}%')
print(f' Test Acc (Standard):  {test_acc*100:.2f}%')
print(f' Test Acc (TTA):       {tta_acc*100:.2f}%')
print(f'')
print(f' === COMPARISON ===')
print(f' v1 (Bug - rescale=1/255, 100x100): 15.58% (model collapse!)')
print(f' v2 (Fixed):                         {test_acc*100:.2f}%')
print(f' vs Method 2 (CBAM CNN + FER2013):   64.22%')
print(f' -> Improvement vs Method 2: {(test_acc - 0.6422)*100:+.2f}%')
print(f' -> Improvement vs v1:       {(test_acc - 0.1558)*100:+.2f}%')
print('=' * 65)